# HiFi-Codec Fine-tuning on Indic Languages
Real AcademiCodec model, codebook-only training

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler

import librosa
import numpy as np
import soundfile as sf
from pesq import pesq
from pystoi import stoi

import os
import json
import glob
import shutil
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
from datetime import datetime
from sklearn.model_selection import train_test_split

import wandb
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(device)}")
    print(f"Memory: {torch.cuda.get_device_properties(device).total_memory / 1e9:.1f} GB")

In [ ]:
CONFIG = {
    'data': {
        'audio_sr': 24000,
        'eval_sr': 16000,
        'clip_duration_sec': 4,
        'train_manifest': 'data/train_manifest.jsonl',
        'val_manifest': 'data/val_manifest.jsonl',
        'test_manifest': 'data/test_manifest.jsonl',
    },
    'model': {
        'config_path': 'AcademiCodec/egs/HiFi-Codec-24k-320d/config_24k_320d.json',
        'ckpt_path': None,  # Set to path if available, else will train from scratch
    },
    'training': {
        'batch_size': 16,
        'num_epochs': 10,
        'learning_rate': 1e-4,
        'max_grad_norm': 1.0,
        'early_stopping_patience': 3,
        'validation_interval': 500,
        'mixed_precision': True,
    },
    'loss': {
        'reconstruction_weight': 1.0,
        'vq_weight': 0.25,
        'indic_phoneme_weight': 0.5,
    },
    'monitoring': {
        'use_wandb': True,
        'wandb_project': 'hifi-codec-indic',
    },
}

os.makedirs('data', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
print(f"Config loaded")

## Data preparation

In [ ]:
def preprocess_audio(audio_path, target_sr=24000):
    try:
        y, sr = librosa.load(audio_path, sr=target_sr, mono=True)
        y, _ = librosa.effects.trim(y, top_db=40)
        y = y / (np.max(np.abs(y)) + 1e-8)
        y = y * 10 ** (-1 / 20)
        return y, sr
    except:
        return None, None

def get_language(path):
    parts = path.split('/')
    for p in parts:
        if len(p) == 2 and p.isalpha():
            return p.lower()
    return 'unknown'

def create_manifest(audio_dir, output_manifest, target_sr=24000):
    data = []
    audio_files = glob.glob(os.path.join(audio_dir, '**/*.wav'), recursive=True)
    audio_files += glob.glob(os.path.join(audio_dir, '**/*.mp3'), recursive=True)
    
    for path in tqdm(audio_files, desc="Creating manifest"):
        try:
            y, sr = librosa.load(path, sr=target_sr, mono=True)
            if y is None:
                continue
            duration = len(y) / target_sr
            if duration < 4:
                continue
            
            data.append({
                'audio_path': path,
                'duration_sec': duration,
                'language': get_language(path),
                'speaker_id': os.path.basename(os.path.dirname(path)),
            })
        except:
            pass
    
    os.makedirs(os.path.dirname(output_manifest), exist_ok=True)
    with open(output_manifest, 'w') as f:
        for entry in data:
            f.write(json.dumps(entry) + '\n')
    
    total_hours = sum(e['duration_sec'] for e in data) / 3600
    print(f"Manifest: {len(data)} clips, {total_hours:.1f} hours")
    return data

def split_manifest(manifest_list, train_path, val_path, test_path):
    speakers = defaultdict(list)
    for entry in manifest_list:
        speakers[entry['speaker_id']].append(entry)
    
    speaker_ids = list(speakers.keys())
    train_spk, temp_spk = train_test_split(speaker_ids, train_size=0.85, random_state=42)
    val_spk, test_spk = train_test_split(temp_spk, train_size=0.333, random_state=42)
    
    train_data = [e for spk in train_spk for e in speakers[spk]]
    val_data = [e for spk in val_spk for e in speakers[spk]]
    test_data = [e for spk in test_spk for e in speakers[spk]]
    
    for path, data in [(train_path, train_data), (val_path, val_data), (test_path, test_data)]:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, 'w') as f:
            for e in data:
                f.write(json.dumps(e) + '\n')
    
    print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

def load_manifest(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return data

In [ ]:
class IndicAudioDataset(Dataset):
    def __init__(self, manifest_path, config, augment=True, split='train'):
        self.manifest = load_manifest(manifest_path)
        self.config = config
        self.augment = augment
        self.split = split
        self.sr = config['data']['audio_sr']
        self.clip_samples = int(config['data']['clip_duration_sec'] * self.sr)
        self.manifest = [m for m in self.manifest if m['duration_sec'] >= config['data']['clip_duration_sec']]
        print(f"{split}: {len(self.manifest)} clips")
    
    def __len__(self):
        return len(self.manifest)
    
    def __getitem__(self, idx):
        entry = self.manifest[idx]
        try:
            y, sr = librosa.load(entry['audio_path'], sr=self.sr, mono=True)
            y, _ = librosa.effects.trim(y, top_db=40)
            
            if len(y) < self.clip_samples:
                y = np.pad(y, (0, self.clip_samples - len(y)), mode='constant')
            
            if self.split == 'train' and len(y) > self.clip_samples:
                start = np.random.randint(0, len(y) - self.clip_samples)
                y = y[start:start + self.clip_samples]
            else:
                if len(y) > self.clip_samples:
                    center = len(y) // 2
                    start = center - self.clip_samples // 2
                    y = y[max(0, start):max(0, start) + self.clip_samples]
                if len(y) < self.clip_samples:
                    y = np.pad(y, (0, self.clip_samples - len(y)), mode='constant')
            
            y = y / (np.max(np.abs(y)) + 1e-8)
            y = y * 10 ** (-1 / 20)
            
            if self.augment and self.split == 'train':
                gain = np.random.uniform(-3, 3)
                y = y * 10 ** (gain / 20)
            
            return {'waveform': torch.from_numpy(y).float()}
        except:
            return {'waveform': torch.zeros(self.clip_samples)}

def collate_fn(batch):
    waveforms = torch.stack([item['waveform'] for item in batch])
    return {'waveform': waveforms}

## Load Real HiFiCodec

In [ ]:
import sys
sys.path.insert(0, 'AcademiCodec')

from academicodec.models.hificodec.vqvae import VQVAE
from academicodec.models.hificodec.env import AttrDict

class HiFiCodecFineTuner(nn.Module):
    def __init__(self, config_path, ckpt_path=None):
        super().__init__()
        self.model = VQVAE(config_path, ckpt_path if ckpt_path else config_path, with_encoder=True)
        
        # Freeze encoder and decoder
        for param in self.model.encoder.parameters():
            param.requires_grad = False
        for param in self.model.generator.parameters():
            param.requires_grad = False
        
        # Codebooks remain trainable
        for param in self.model.quantizer.quantizer_modules.parameters():
            param.requires_grad = True
        for param in self.model.quantizer.quantizer_modules2.parameters():
            param.requires_grad = True
        
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    
    def forward(self, waveform):
        # waveform: [B, T]
        latent = self.model.encoder(waveform.unsqueeze(1))  # [B, 512, T]
        quantized, vq_loss, indices = self.model.quantizer(latent)  # Quantize
        reconstructed = self.model.generator(quantized)  # [B, 1, T]
        
        return {
            'reconstructed': reconstructed.squeeze(1),
            'vq_loss': vq_loss,
            'indices': indices,
        }
    
    def get_codebook_stats(self):
        stats = {}
        # Layer 1
        for i, qm in enumerate(self.model.quantizer.quantizer_modules):
            embed = qm.embedding.weight.data
            stats[f'layer1_codebook{i}_norm'] = embed.norm(dim=1).mean().item()
        # Layer 2
        for i, qm in enumerate(self.model.quantizer.quantizer_modules2):
            embed = qm.embedding.weight.data
            stats[f'layer2_codebook{i}_norm'] = embed.norm(dim=1).mean().item()
        return stats

## Loss functions

In [ ]:
def compute_mel_spectrogram(waveform, sr=24000, n_fft=1024, hop_length=240, n_mels=80):
    if waveform.dim() == 2:
        waveform = waveform.unsqueeze(0)
    mel_spec = []
    for w in waveform:
        S = librosa.feature.melspectrogram(
            y=w.detach().cpu().numpy(), sr=sr, n_fft=n_fft,
            hop_length=hop_length, n_mels=n_mels
        )
        mel_spec.append(torch.from_numpy(S).to(waveform.device))
    return torch.log(torch.stack(mel_spec) + 1e-9)

def indic_phoneme_loss(original, reconstructed, sr=24000):
    mel_orig = compute_mel_spectrogram(original, sr=sr)
    mel_recon = compute_mel_spectrogram(reconstructed, sr=sr)
    
    min_time = min(mel_orig.shape[2], mel_recon.shape[2])
    mel_orig = mel_orig[:, :, :min_time]
    mel_recon = mel_recon[:, :, :min_time]
    
    freq_bins = np.linspace(0, 12000, 80)
    aspiration_bins = np.where((freq_bins >= 3000) & (freq_bins <= 8000))[0]
    retroflex_bins = np.where((freq_bins >= 1000) & (freq_bins <= 3000))[0]
    
    base_loss = F.l1_loss(mel_orig, mel_recon)
    
    if len(aspiration_bins) > 0:
        asp_loss = F.l1_loss(mel_orig[:, aspiration_bins, :], mel_recon[:, aspiration_bins, :])
        base_loss += 2.0 * asp_loss
    
    if len(retroflex_bins) > 0:
        ret_loss = F.l1_loss(mel_orig[:, retroflex_bins, :], mel_recon[:, retroflex_bins, :])
        base_loss += 1.5 * ret_loss
    
    return base_loss

def compute_combined_loss(original, reconstructed, vq_loss, config):
    recon_loss = F.l1_loss(original, reconstructed)
    indic_loss = indic_phoneme_loss(original, reconstructed)
    
    total = (
        config['loss']['reconstruction_weight'] * recon_loss +
        config['loss']['vq_weight'] * vq_loss +
        config['loss']['indic_phoneme_weight'] * indic_loss
    )
    
    return total, {'recon': recon_loss, 'vq': vq_loss, 'indic': indic_loss}

## Evaluation metrics

In [ ]:
def compute_pesq_score(orig, recon, sr=16000):
    try:
        if isinstance(orig, torch.Tensor):
            orig = orig.detach().cpu().numpy()
        if isinstance(recon, torch.Tensor):
            recon = recon.detach().cpu().numpy()
        return pesq(sr, orig, recon, 'wb')
    except:
        return np.nan

def compute_stoi_score(orig, recon, sr=16000):
    try:
        if isinstance(orig, torch.Tensor):
            orig = orig.detach().cpu().numpy()
        if isinstance(recon, torch.Tensor):
            recon = recon.detach().cpu().numpy()
        return stoi(orig, recon, sr)
    except:
        return np.nan

def compute_si_sdr(orig, recon):
    if isinstance(orig, torch.Tensor):
        orig = orig.detach().cpu().numpy()
    if isinstance(recon, torch.Tensor):
        recon = recon.detach().cpu().numpy()
    
    orig = orig / (np.linalg.norm(orig) + 1e-8)
    alpha = np.sum(orig * recon) / (np.sum(orig ** 2) + 1e-8)
    s_target = alpha * orig
    e_noise = recon - s_target
    si_sdr = 10 * np.log10(np.sum(s_target ** 2) / (np.sum(e_noise ** 2) + 1e-8) + 1e-8)
    return si_sdr

## Trainer

In [ ]:
class Trainer:
    def __init__(self, model, config, device):
        self.model = model.to(device)
        self.config = config
        self.device = device
        self.optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=config['training']['learning_rate']
        )
        self.scaler = GradScaler() if config['training']['mixed_precision'] else None
        
        if config['monitoring']['use_wandb']:
            wandb.init(
                project=config['monitoring']['wandb_project'],
                config=config,
                name=f"hifi-indic-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
            )
        
        self.best_val_loss = float('inf')
        self.early_stop_count = 0
    
    def train_epoch(self, train_loader, epoch):
        self.model.train()
        total_loss = 0.0
        
        for batch_idx, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
            waveform = batch['waveform'].to(self.device)
            self.optimizer.zero_grad()
            
            if self.scaler:
                with autocast():
                    output = self.model(waveform)
                    loss, losses = compute_combined_loss(
                        waveform, output['reconstructed'], output['vq_loss'], self.config
                    )
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
            else:
                output = self.model(waveform)
                loss, losses = compute_combined_loss(
                    waveform, output['reconstructed'], output['vq_loss'], self.config
                )
                loss.backward()
            
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, self.model.parameters()),
                self.config['training']['max_grad_norm']
            )
            
            if self.scaler:
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                self.optimizer.step()
            
            total_loss += loss.item()
            
            if batch_idx % 50 == 0 and self.config['monitoring']['use_wandb']:
                log_dict = {'train/loss': loss.item()}
                for k, v in losses.items():
                    if isinstance(v, torch.Tensor):
                        log_dict[f'train/loss_{k}'] = v.item()
                
                cb_stats = self.model.get_codebook_stats()
                for k, v in cb_stats.items():
                    log_dict[f'codebook/{k}'] = v
                
                wandb.log(log_dict)
        
        return total_loss / len(train_loader)
    
    def validate(self, val_loader, epoch):
        self.model.eval()
        total_loss = 0.0
        metrics = {'pesq': [], 'stoi': [], 'si_sdr': []}
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validating"):
                waveform = batch['waveform'].to(self.device)
                output = self.model(waveform)
                loss, _ = compute_combined_loss(
                    waveform, output['reconstructed'], output['vq_loss'], self.config
                )
                total_loss += loss.item()
                
                for i in range(waveform.shape[0]):
                    orig_16k = librosa.resample(
                        waveform[i].cpu().numpy(),
                        orig_sr=self.config['data']['audio_sr'],
                        target_sr=self.config['data']['eval_sr']
                    )
                    recon_16k = librosa.resample(
                        output['reconstructed'][i].cpu().numpy(),
                        orig_sr=self.config['data']['audio_sr'],
                        target_sr=self.config['data']['eval_sr']
                    )
                    
                    pesq_val = compute_pesq_score(orig_16k, recon_16k)
                    stoi_val = compute_stoi_score(orig_16k, recon_16k)
                    si_sdr_val = compute_si_sdr(orig_16k, recon_16k)
                    
                    if not np.isnan(pesq_val):
                        metrics['pesq'].append(pesq_val)
                    if not np.isnan(stoi_val):
                        metrics['stoi'].append(stoi_val)
                    if not np.isnan(si_sdr_val):
                        metrics['si_sdr'].append(si_sdr_val)
        
        avg_val_loss = total_loss / len(val_loader)
        
        if avg_val_loss < self.best_val_loss:
            self.best_val_loss = avg_val_loss
            self.early_stop_count = 0
            self.save_checkpoint(epoch)
        else:
            self.early_stop_count += 1
        
        if self.config['monitoring']['use_wandb']:
            log_dict = {'val/loss': avg_val_loss}
            for k, vals in metrics.items():
                if vals:
                    log_dict[f'val/{k}'] = np.mean(vals)
            wandb.log(log_dict)
        
        return avg_val_loss
    
    def save_checkpoint(self, epoch):
        path = f'checkpoints/best_hifi_indic.pt'
        torch.save({
            'epoch': epoch,
            'model': self.model.state_dict(),
            'optimizer': self.optimizer.state_dict(),
        }, path)
        print(f"Saved: {path}")
    
    def should_stop(self):
        return self.early_stop_count >= self.config['training']['early_stopping_patience']

## Generate sample data & manifest

In [ ]:
os.makedirs('data/sample_audio', exist_ok=True)

for lang in ['hi', 'ta', 'te', 'bn', 'pa']:
    lang_dir = f'data/sample_audio/{lang}'
    os.makedirs(lang_dir, exist_ok=True)
    
    for i in range(10):
        duration = np.random.uniform(4, 8)
        sr = 24000
        t = np.linspace(0, duration, int(sr * duration))
        
        f0 = np.random.uniform(80, 250)
        signal = np.sin(2 * np.pi * f0 * t)
        for h in range(2, 5):
            signal += 0.3 * np.sin(2 * np.pi * f0 * h * t) / h
        signal += 0.1 * np.random.randn(len(signal))
        
        signal = signal / (np.max(np.abs(signal)) + 1e-8) * 0.95
        sf.write(f'{lang_dir}/speaker_{i:03d}.wav', signal, sr)

print(f"Created 50 sample audio files")

In [ ]:
all_data = create_manifest('data/sample_audio', 'data/all_manifest.jsonl')
split_manifest(
    all_data,
    CONFIG['data']['train_manifest'],
    CONFIG['data']['val_manifest'],
    CONFIG['data']['test_manifest']
)

## Load data & model

In [ ]:
train_dataset = IndicAudioDataset(CONFIG['data']['train_manifest'], CONFIG, augment=True, split='train')
val_dataset = IndicAudioDataset(CONFIG['data']['val_manifest'], CONFIG, augment=False, split='val')

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG['training']['batch_size'], shuffle=True,
    num_workers=4, collate_fn=collate_fn, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=CONFIG['training']['batch_size'], shuffle=False,
    num_workers=4, collate_fn=collate_fn, pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

In [ ]:
model = HiFiCodecFineTuner(
    CONFIG['model']['config_path'],
    CONFIG['model']['ckpt_path']
)
model = model.to(device)

## Train

In [ ]:
trainer = Trainer(model, CONFIG, device)

train_losses = []
val_losses = []

for epoch in range(CONFIG['training']['num_epochs']):
    print(f"\n{'='*60}\nEpoch {epoch+1}/{CONFIG['training']['num_epochs']}\n{'='*60}")
    
    train_loss = trainer.train_epoch(train_loader, epoch)
    train_losses.append(train_loss)
    
    val_loss = trainer.validate(val_loader, epoch)
    val_losses.append(val_loss)
    
    print(f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | Best: {trainer.best_val_loss:.4f}")
    
    if trainer.should_stop():
        print(f"Early stop at epoch {epoch+1}")
        break

if CONFIG['monitoring']['use_wandb']:
    wandb.finish()

print(f"\n{'='*60}\nTraining Complete\n{'='*60}")

## Visualize

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, 'b-o', label='Train', linewidth=2)
axes[0].plot(val_losses, 'r-s', label='Val', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

components = ['Recon', 'VQ', 'Indic']
weights = [
    CONFIG['loss']['reconstruction_weight'],
    CONFIG['loss']['vq_weight'],
    CONFIG['loss']['indic_phoneme_weight'],
]
axes[1].bar(components, weights, color=['#FF6B6B', '#4ECDC4', '#98D8C8'], alpha=0.8)
axes[1].set_ylabel('Weight')
axes[1].set_title('Loss Components')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
print("Saved: training_curves.png")
plt.show()

## Test evaluation

In [ ]:
checkpoint = torch.load('checkpoints/best_hifi_indic.pt', map_location=device)
model.load_state_dict(checkpoint['model'])
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")

test_dataset = IndicAudioDataset(CONFIG['data']['test_manifest'], CONFIG, augment=False, split='test')
test_loader = DataLoader(
    test_dataset, batch_size=CONFIG['training']['batch_size'], shuffle=False,
    num_workers=4, collate_fn=collate_fn, pin_memory=True
)

model.eval()
results = {'pesq': [], 'stoi': [], 'si_sdr': [], 'loss': []}

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        waveform = batch['waveform'].to(device)
        output = model(waveform)
        loss, _ = compute_combined_loss(
            waveform, output['reconstructed'], output['vq_loss'], CONFIG
        )
        results['loss'].append(loss.item())
        
        for i in range(waveform.shape[0]):
            orig_16k = librosa.resample(
                waveform[i].cpu().numpy(), orig_sr=24000, target_sr=16000
            )
            recon_16k = librosa.resample(
                output['reconstructed'][i].cpu().numpy(), orig_sr=24000, target_sr=16000
            )
            
            pesq_val = compute_pesq_score(orig_16k, recon_16k)
            stoi_val = compute_stoi_score(orig_16k, recon_16k)
            si_sdr_val = compute_si_sdr(orig_16k, recon_16k)
            
            if not np.isnan(pesq_val):
                results['pesq'].append(pesq_val)
            if not np.isnan(stoi_val):
                results['stoi'].append(stoi_val)
            if not np.isnan(si_sdr_val):
                results['si_sdr'].append(si_sdr_val)

print("\nTest Results:")
for metric in ['pesq', 'stoi', 'si_sdr', 'loss']:
    if results[metric]:
        print(f"{metric.upper()}: {np.mean(results[metric]):.4f} ± {np.std(results[metric]):.4f}")